# OpenOppsDB — Starter

Front door for the public OpenOppsDB snapshot. Read-only, credential-free, attached only to `wyattowalsh/openoppsdb`. Use `%%sql` for the first queries, then continue in Explorer or the SQL playground.

## Contents

- Setup (read-only `/kaggle/input`, `mode=ro&immutable=1`)
- Queries and charts for this kernel
- Links to the rest of the collection

## Collection

| Notebook | Kernel | What it is for |
| --- | --- | --- |
| **Starter** | [`wyattowalsh/openoppsdb-starter-notebook`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-starter-notebook) | Front door: tables, recent open jobs, first `%%sql` cells |
| **Explorer (featured)** | [`wyattowalsh/openoppsdb-explorer`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-explorer) | Gradio UI: jobs, companies, skills, filters/plots |
| **Advanced usage** | [`wyattowalsh/openoppsdb-advanced-usage`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-advanced-usage) | Joins, version history, company drill-down, Parquet |
| **SQL playground** | [`wyattowalsh/openoppsdb-sql-playground`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-sql-playground) | JupySQL studio: CTEs, DuckDB attach, Parquet scans |
| **Hiring market map** | [`wyattowalsh/openoppsdb-hiring-market-map`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-hiring-market-map) | Company, provider, location, and remote mix charts |
| **Skills radar** | [`wyattowalsh/openoppsdb-skills-radar`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-skills-radar) | Skill groups, keywords, and co-occurrence |
| **Snapshot health** | [`wyattowalsh/openoppsdb-snapshot-health`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-snapshot-health) | Coverage, freshness, sync runs, observation mix |


In [ ]:
%pip install -q jupysql==0.11.1 duckdb==1.5.5 duckdb-engine==0.17.0 plotly==7.0.0


In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"
ROUTE_LEDGER = {
    "pine": "#2f6f50",
    "paper": "#f7f1df",
    "brass": "#d99629",
    "ink": "#1d281f",
    "info": "#336d8f",
}

db_candidates = sorted(Path("/kaggle/input").glob("**/openoppsdb.sqlite"))
if not db_candidates:
    raise FileNotFoundError("No openoppsdb.sqlite input found under /kaggle/input")
DB_PATH = db_candidates[0]
DATASET_DIR = DB_PATH.parent
DB_URI = f"file:{DB_PATH}?mode=ro&immutable=1"
PARQUET_DIR = DATASET_DIR / "exports" / "parquet"
print(f"Reading OpenOppsDB snapshot from {DB_PATH}")


In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 25
%config SqlMagic.autolimit = 100


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(
    "ATTACH ? AS oo (TYPE SQLITE, READ_ONLY)",
    [f"file:{DB_PATH}?mode=ro&immutable=1"],
)
print("DuckDB attached the read-only SQLite snapshot as oo")


In [ ]:
%sql con --alias openopps


In [ ]:
%%sql
SELECT table_name, table_title, table_description
FROM oo.openopps_tables
ORDER BY table_name


In [ ]:
%%sql
SELECT job_id, title, company, locations, employment_type,
       first_seen_at, last_seen_at, posting_url
FROM oo.job_versions
ORDER BY last_seen_at DESC
LIMIT 20


In [ ]:
with sqlite3.connect(DB_URI, uri=True) as conn:
    tables = pd.read_sql_query(
        "select table_name from openopps_tables order by table_name",
        conn,
    )
    counts = pd.DataFrame(
        {
            "table": tables["table_name"],
            "rows": [
                conn.execute(f'select count(*) from "{name}"').fetchone()[0]
                for name in tables["table_name"]
            ],
        }
    )
focus = counts[counts["table"].isin(
    ["jobs", "job_versions", "job_sync_runs", "sources", "boards"]
)]
fig = px.bar(
    focus,
    x="table",
    y="rows",
    color_discrete_sequence=[ROUTE_LEDGER["pine"]],
    title="Core table row counts",
)
fig.update_layout(paper_bgcolor=ROUTE_LEDGER["paper"], font_color=ROUTE_LEDGER["ink"])
fig.show()
focus
